# 10 — Text Clustering
**Goal:** Agrupar documentos similares sin etiquetas — descubrimiento de estructura.

```
K-Means         → clusters esféricos sobre vectores TF-IDF
Agglomerative   → jerarquía de clusters (dendrogram)
DBSCAN          → clusters de densidad, robusto a outliers
```

Diferencia clave con topic modeling (LDA): clustering asigna cada documento a **un único** cluster; LDA le da una **mezcla** de temas.

In [ ]:
import os
os.chdir('/Volumes/SSD_Gabo/proyectos/growth-analytics')

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer
from sklearn.pipeline import Pipeline
from sklearn.metrics import silhouette_score, adjusted_rand_score
from scipy.cluster.hierarchy import dendrogram, linkage

plt.rcParams.update({
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e8e8e8', 'axes.axisbelow': True,
})

STOPWORDS_ES = [
    'a','al','como','con','de','del','el','en','es','la','las','lo','los',
    'me','mi','muy','no','o','para','por','que','se','si','sin','su','te',
    'todo','un','una','y','yo','tuve','llevo',
]

def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^\wáéíóúüñ\s]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

In [ ]:
topic_templates = {
    'app_ux':             ['La app está muy lenta y tarda en cargar', 'La aplicación no carga, da error al abrir', 'La app se cierra sola al iniciar sesión', 'Muy lenta la aplicación, tarda en responder', 'La interfaz es confusa y difícil de usar'],
    'otp_seguridad':      ['No me llegó el OTP al celular', 'El código OTP no llega por SMS', 'El código de verificación no aparece', 'Necesito el OTP pero no llega', 'El SMS con el código no llega nunca'],
    'documentos':         ['La carga de documentos falla constantemente', 'No puedo subir los documentos requeridos', 'El proceso de carga da error todo el tiempo', 'Los documentos no se suben al sistema', 'No acepta mi documento de identidad'],
    'credito_evaluacion': ['Llevo días esperando la evaluación crediticia', 'Mi solicitud lleva semanas en evaluación', 'No hay información sobre mi evaluación', 'Cuándo aprueban mi solicitud de tarjeta', 'La evaluación no termina, mucho tiempo'],
    'soporte':            ['El call center no responde, llevo horas', 'Nadie del soporte responde mis mensajes', 'Pésima atención, nadie resuelve mis problemas', 'Llamé cuatro veces sin resolución', 'El chat no funciona y el teléfono no contesta'],
    'positivo':           ['Excelente servicio, aprobaron muy rápido', 'Proceso de solicitud fácil y eficiente', 'Increíbles beneficios, recomiendo la tarjeta', 'Muy contento, todo salió perfecto', 'Activación sencilla y proceso rápido'],
}

rng = np.random.default_rng(42)
corpus, true_labels = [], []
for label, templates in topic_templates.items():
    for _ in range(80):
        corpus.append(templates[rng.integers(len(templates))])
        true_labels.append(label)

df = pd.DataFrame({'text': corpus, 'true_label': true_labels})
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
print(f'Corpus: {len(df)} docs | {df.true_label.nunique()} temas reales')

## 1. Vectorizar — TF-IDF + reducción SVD

In [ ]:
# Pipeline estándar para clustering de texto:
# TF-IDF → TruncatedSVD (LSA) → L2-normalize → K-Means
# La normalización L2 hace que K-Means minimice similitud coseno

tfidf = TfidfVectorizer(preprocessor=preprocess, stop_words=STOPWORDS_ES,
                         ngram_range=(1,2), min_df=2, sublinear_tf=True)
X_tfidf = tfidf.fit_transform(df['text'])   # sparse (480, vocab)

# Reducir a 50 dimensiones con SVD (LSA)
svd = TruncatedSVD(n_components=50, random_state=42)
X_svd = svd.fit_transform(X_tfidf)          # dense (480, 50)

# L2-normalize para que la distancia euclídea ≈ distancia coseno
X_norm = Normalizer().fit_transform(X_svd)  # (480, 50)

print(f'TF-IDF shape: {X_tfidf.shape}')
print(f'SVD shape:    {X_svd.shape}')
print(f'Varianza explicada por 50 componentes: {svd.explained_variance_ratio_.sum():.1%}')

## 2. Elegir K — método del codo y silhouette

In [ ]:
inertias    = []
silhouettes = []
ks = range(2, 12)

for k in ks:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_norm)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_norm, labels))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(list(ks), inertias, 'o-', color='#4361ee', linewidth=2)
axes[0].axvline(6, color='#f72585', linestyle='--', label='k=6 (real)')
axes[0].set_xlabel('k'); axes[0].set_ylabel('Inercia (SSE)')
axes[0].set_title('Elbow Method'); axes[0].legend()

axes[1].plot(list(ks), silhouettes, 'o-', color='#06d6a0', linewidth=2)
axes[1].axvline(6, color='#f72585', linestyle='--', label='k=6 (real)')
axes[1].set_xlabel('k'); axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score (mayor = mejor separación)'); axes[1].legend()

plt.tight_layout()
plt.show()

## 3. K-Means con k=6

In [ ]:
km = KMeans(n_clusters=6, random_state=42, n_init=20)
df['cluster'] = km.fit_predict(X_norm)

# Evaluar vs etiquetas reales
ari = adjusted_rand_score(df['true_label'], df['cluster'])
sil = silhouette_score(X_norm, df['cluster'])
print(f'Adjusted Rand Index: {ari:.3f}  (1.0 = perfecto, 0 = aleatorio)')
print(f'Silhouette Score:    {sil:.3f}  (mayor = clusters más compactos)\n')

# Alineación cluster ↔ tema real
print(pd.crosstab(df['true_label'], df['cluster']))

In [ ]:
# Top palabras por cluster (centroides en espacio TF-IDF original)
feature_names = tfidf.get_feature_names_out()

# Proyectar centroides de vuelta al espacio TF-IDF
# centroids_svd shape: (6, 50) → proyectar con SVD.T
centroids_svd  = km.cluster_centers_                          # (6, 50)
centroids_tfidf = centroids_svd @ svd.components_            # (6, vocab)

print('Top palabras por cluster K-Means:')
for i, centroid in enumerate(centroids_tfidf):
    top_idx = centroid.argsort()[::-1][:8]
    top_words = [feature_names[j] for j in top_idx]
    print(f'  Cluster {i}: {" | ".join(top_words)}')

## 4. Clustering jerárquico — dendrogram

In [ ]:
# Usar una muestra pequeña para el dendrograma
sample_idx = []
for label in df['true_label'].unique():
    idxs = df[df['true_label'] == label].index[:8]
    sample_idx.extend(idxs)

X_sample = X_norm[sample_idx]
sample_labels = df['true_label'].iloc[sample_idx].tolist()

# Linkage matrix
Z = linkage(X_sample, method='ward')

fig, ax = plt.subplots(figsize=(14, 5))
dendrogram(
    Z,
    labels=sample_labels,
    leaf_rotation=45,
    leaf_font_size=8,
    color_threshold=1.2,
    ax=ax,
)
ax.set_title('Dendrograma jerárquico — Ward linkage')
ax.set_ylabel('Distancia')
plt.tight_layout()
plt.show()

## 5. Visualización 2D — proyección de clusters

In [ ]:
# Proyectar a 2D para visualizar
svd2 = TruncatedSVD(n_components=2, random_state=42)
X2d = svd2.fit_transform(X_tfidf)

COLORS = ['#4361ee','#f72585','#06d6a0','#ff9f1c','#7209b7','#e63946']
TOPICS = sorted(df['true_label'].unique())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Por tema real
for i, topic in enumerate(TOPICS):
    mask = df['true_label'] == topic
    axes[0].scatter(X2d[mask, 0], X2d[mask, 1], c=COLORS[i], label=topic, alpha=0.5, s=25)
axes[0].set_title('Temas REALES'); axes[0].legend(fontsize=7, loc='upper right')

# Por cluster K-Means
for k in range(6):
    mask = df['cluster'] == k
    axes[1].scatter(X2d[mask, 0], X2d[mask, 1], c=COLORS[k], label=f'Cluster {k}', alpha=0.5, s=25)
axes[1].set_title('Clusters K-MEANS'); axes[1].legend(fontsize=7, loc='upper right')

plt.suptitle('Proyección 2D (TF-IDF + SVD) — comparación', fontweight='bold')
plt.tight_layout()
plt.show()

## Resumen
| Algoritmo | Cuándo usar |
|---|---|
| K-Means | Default; escala bien, asume clusters esféricos |
| Agglomerative | Cuando necesitas jerarquía; lento en datasets grandes |
| DBSCAN | Cuando hay outliers o clusters de forma irregular |

| Métrica | Qué mide |
|---|---|
| Silhouette | Cohesión intra-cluster vs separación inter-cluster |
| Inercia (SSE) | Suma de distancias al centroide — elbow method |
| Adjusted Rand Index | Comparar clustering vs etiquetas reales |

| Pipeline recomendado | |
|---|---|
| `TF-IDF → TruncatedSVD(50) → Normalizer → KMeans` | Estándar para texto |

**Siguiente:** `11_embeddings_lsa.ipynb` — word vectors mediante factorización matricial.